# Laboratorio #3 - Parte C: Adicionales (Regresión, Coseno, 3D, Kalman, Complementario)
## INF0157 - Sistemas

**Autor:** Renato Carrasco  
**Objetivo:** Analizar regresión lineal, similitud coseno, visualización 3D, filtro Kalman y filtro complementario

## 1. Importar librerías

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from src.data_loader import DataLoader
from src.regression.linear_regression import LinearRegressionModel
from src.similarity.cosine_distance import CosineSimilarity, print_cosine_analysis
from src.filters.kalman_filter import KalmanFilter
from src.filters.complementary_filter import ComplementaryFilter
from src.utils.visualization import Visualizer

print("✅ Librerías importadas correctamente")

# SECCIÓN 3: Regresión Lineal

## 3.1 Datos de Altura vs Peso

In [ ]:
# Tabla de Pesos y Alturas
pesos = np.array([60.0, 65.0, 72.3, 75.0, 80.0])  # kg
alturas = np.array([1.60, 1.65, 1.70, 1.73, 1.80])  # m

# Crear DataFrame
data_regression = pd.DataFrame({
    'Peso (kg)': pesos,
    'Altura (m)': alturas
})

print("Datos de Peso y Altura:")
print(data_regression)

# Preparar datos para regresión
X = alturas.reshape(-1, 1)
y = pesos

## 3.2 Entrenar modelo de regresión lineal

In [ ]:
# Crear y entrenar modelo
regression_model = LinearRegressionModel()
regression_model.fit(X, y)

print("✅ Modelo de regresión lineal entrenado")
print(f"\nEcuación del modelo:")
print(regression_model.get_equation())

## 3.3 Predicciones en set de test

In [ ]:
# Datos de test
alturas_test = np.array([1.58, 1.62, 1.69, 1.76, 1.82])  # m
pesos_test = np.array([58.0, 63.0, 70.0, 78.0, 85.0])  # kg

X_test = alturas_test.reshape(-1, 1)
y_test = pesos_test

# Realizar predicciones
predictions = regression_model.predict(X_test)

print("Predicciones en set de test:")
print(f"\n{'Altura (m)':<12} {'Peso Real (kg)':<18} {'Peso Predicho (kg)':<20} {'Error (kg)'}")
print("="*65)
for altura, peso_real, peso_pred in zip(alturas_test, pesos_test, predictions):
    error = peso_real - peso_pred
    print(f"{altura:<12.2f} {peso_real:<18.2f} {peso_pred:<20.2f} {error:>10.2f}")

## 3.4 Evaluación del modelo

In [ ]:
# Evaluar modelo
metrics = regression_model.evaluate(X_test, y_test)

print("\nMétricas de evaluación:")
print(f"  MSE (Mean Squared Error): {metrics['mse']:.4f}")
print(f"  RMSE (Root Mean Squared Error): {metrics['rmse']:.4f}")
print(f"  MAE (Mean Absolute Error): {metrics['mae']:.4f}")
print(f"  R² Score: {metrics['r2']:.4f}")

# Calcular RSS
residuals = y_test - predictions
rss = np.sum(residuals**2)
print(f"\n  RSS (Residual Sum of Squares): {rss:.4f}")

## 3.5 Visualizar regresión

In [ ]:
# Graficar regresión
equation = regression_model.get_equation()
Visualizer.plot_regression(X, y, X_test, y_test, predictions, equation)
plt.show()

# SECCIÓN 4: Distancia del Coseno

## 4.1 Definir vectores

In [ ]:
# Vectores según el enunciado
A = np.array([2, 1, 0, 2, 0, 1, 1, 1])
B = np.array([2, 1, 1, 1, 0, 1, 1])
P = np.array([1, 2, 3, 0, 4, 6, 7, 9])
Q = np.array([2, 4, 3, 1, 8, 2, 4, 1])
S = np.array([2, 1, 4, 7, 1, 4, 5, 6])
T = np.array([3, 3, 3, 6, 1, 7, 1, 8])

print("Vectores definidos:")
print(f"A = {A}")
print(f"B = {B}")
print(f"P = {P}")
print(f"Q = {Q}")
print(f"S = {S}")
print(f"T = {T}")

## 4.2 Calcular similitud del coseno

In [ ]:
# Crear diccionario de vectores con longitudes compatibles
# Los adaptamos para que tengan la misma longitud
max_len = max(len(A), len(B), len(P), len(Q), len(S), len(T))

def pad_vector(v, length):
    """Rellena un vector con ceros hasta la longitud especificada."""
    if len(v) < length:
        return np.concatenate([v, np.zeros(length - len(v))])
    return v

vectors_dict = {
    'A': pad_vector(A, max_len),
    'B': pad_vector(B, max_len),
    'P': pad_vector(P, max_len),
    'Q': pad_vector(Q, max_len),
    'S': pad_vector(S, max_len),
    'T': pad_vector(T, max_len),
}

# Imprimir análisis
print_cosine_analysis(vectors_dict)

## 4.3 Interpretación de resultados

In [ ]:
print("\n" + "="*60)
print("INTERPRETACIÓN DE RESULTADOS - DISTANCIA DEL COSENO")
print("="*60)

print("\n❓ Respuestas a las preguntas:")

print("\n1️⃣  ¿Cuáles son los ángulos de las distancias anteriores?")

# Calcular pares específicos
cos_AB = CosineSimilarity.cosine_distance(A, B)
angle_AB = CosineSimilarity.angle_from_cosine_degrees(cos_AB)
print(f"   - Ángulo entre A y B: {angle_AB:.2f}°")

cos_PQ = CosineSimilarity.cosine_distance(P, Q)
angle_PQ = CosineSimilarity.angle_from_cosine_degrees(cos_PQ)
print(f"   - Ángulo entre P y Q: {angle_PQ:.2f}°")

cos_ST = CosineSimilarity.cosine_distance(S, T)
angle_ST = CosineSimilarity.angle_from_cosine_degrees(cos_ST)
print(f"   - Ángulo entre S y T: {angle_ST:.2f}°")

print("\n2️⃣  Cuando θ = 0 rad (0°): Los vectores son idénticos (paralelos en la misma dirección)")
print("   - Significa: Máxima similitud, cos(θ) = 1")

print("\n3️⃣  Cuando θ = π rad (180°): Los vectores son opuestos (antiparalelos)")
print("   - Significa: Mínima similitud, cos(θ) = -1")

print("\n4️⃣  Cuando θ = π/2 rad (90°): Los vectores son ortogonales")
print("   - Significa: No hay similitud, cos(θ) = 0")

print("\n" + "="*60)

# SECCIÓN 5: Visualización 3D con Anaconda

## 5.1 Cargar y visualizar puntos 3D

In [ ]:
# Cargar puntos 3D
df_3d = DataLoader.load_3d_points()

print("Puntos 3D cargados:")
print(f"  Forma: {df_3d.shape}")
print(f"  Columnas: {list(df_3d.columns)}")
print(f"\nPrimeras 5 filas:")
print(df_3d.head())

# Extraer coordenadas
X_3d = df_3d[['x', 'y', 'z']].values

## 5.2 Visualizar sin centroide

In [ ]:
# Visualizar sin centroide
Visualizer.plot_3d_points(X_3d, title="Puntos 3D - Sin Centroide")
plt.show()

## 5.3 Calcular y visualizar centroide

In [ ]:
# Calcular centroide
centroide = np.mean(X_3d, axis=0)
print(f"Centroide calculado: {centroide}")
print(f"  x: {centroide[0]:.4f}")
print(f"  y: {centroide[1]:.4f}")
print(f"  z: {centroide[2]:.4f}")

# Visualizar con centroide
Visualizer.plot_3d_points(X_3d, centroids=centroide.reshape(1, -1), 
                          title="Puntos 3D - Con Centroide (Color Azul)")
plt.show()

# SECCIÓN 7: Filtro de Kalman

## 7.1 Generar trayectoria sintética

In [ ]:
# Parámetros de la trayectoria
dt = 0.1  # Intervalo de tiempo
t = np.arange(0, 100, dt)  # Tiempo de 0 a 100 segundos

# Trayectoria real: Movimiento acelerado
real_trajectory = 0.1 * (t**2) - t

# Agregamos ruido a la trayectoria real
np.random.seed(42)
noise = np.random.normal(0, 50, len(t))  # Ruido gaussiano
measurements = real_trajectory + noise

print(f"Trayectoria generada:")
print(f"  Número de mediciones: {len(measurements)}")
print(f"  Tiempo total: {t[-1]:.1f}s")
print(f"  Valor mínimo: {measurements.min():.2f}")
print(f"  Valor máximo: {measurements.max():.2f}")

## 7.2 Aplicar filtro de Kalman

In [ ]:
# Crear y aplicar filtro de Kalman
kf = KalmanFilter(dt=dt, process_variance=1e-5, measurement_variance=50**2,
                  initial_position=measurements[0], initial_velocity=0)

estimates = kf.filter(measurements)

print("✅ Filtro de Kalman aplicado")
print(f"  Número de estimaciones: {len(estimates)}")

## 7.3 Visualizar resultados del filtro

In [ ]:
# Graficar resultados
Visualizer.plot_kalman_filter(measurements, estimates, real_trajectory)
plt.show()

## 7.4 Análisis del error

In [ ]:
# Calcular errores
error_measurements = np.abs(measurements - real_trajectory)
error_estimates = np.abs(estimates - real_trajectory)

print("\nAnálisis de errores del Filtro de Kalman:")
print(f"\nError sin filtro (mediciones):")
print(f"  Error medio: {np.mean(error_measurements):.2f}")
print(f"  Error máximo: {np.max(error_measurements):.2f}")
print(f"  Error RMS: {np.sqrt(np.mean(error_measurements**2)):.2f}")

print(f"\nError con filtro de Kalman:")
print(f"  Error medio: {np.mean(error_estimates[1:]):.2f}")
print(f"  Error máximo: {np.max(error_estimates[1:]):.2f}")
print(f"  Error RMS: {np.sqrt(np.mean(error_estimates[1:]**2)):.2f}")

mejora = (1 - np.mean(error_estimates[1:]) / np.mean(error_measurements)) * 100
print(f"\n📊 Mejora del filtro: {mejora:.1f}%")

# SECCIÓN 8: Filtro Complementario

## 8.1 Generar mediciones de sensores

In [ ]:
# Sensor 1: Muy ruidoso pero sin drift
measurements_noisy = real_trajectory + np.random.normal(0, 20, len(t))

# Sensor 2: Integración acumulativa (tiene drift pero suave)
velocity_true = 0.2 * t - 1  # Velocidad verdadera
measurements_integrated = np.cumsum(velocity_true) * dt + np.random.normal(0, 5, len(t))

print("Mediciones de sensores generadas:")
print(f"  Sensor ruidoso - Media: {np.mean(measurements_noisy):.2f}")
print(f"  Sensor integrado - Media: {np.mean(measurements_integrated):.2f}")

## 8.2 Aplicar filtro complementario

# Crear y aplicar filtro complementario
cf = ComplementaryFilter(alpha=0.95, dt=dt)
estimates_cf = cf.filter_series(measurements_noisy, measurements_integrated)

print("✅ Filtro complementario aplicado")
print(f"  Alpha: {cf.get_alpha()}")

## 8.3 Visualizar resultados del filtro complementario

In [ ]:
# Graficar resultados
Visualizer.plot_complementary_filter(measurements_noisy, measurements_integrated, 
                                    estimates_cf, real_trajectory)
plt.show()

## 8.4 Analizar efecto del alpha

In [ ]:
# Probar diferentes valores de alpha
alphas = [0.5, 0.7, 0.9, 0.95]

plt.figure(figsize=(14, 8))

for i, alpha in enumerate(alphas, 1):
    cf_test = ComplementaryFilter(alpha=alpha)
    estimates_test = cf_test.filter_series(measurements_noisy, measurements_integrated)
    
    error = np.sqrt(np.mean((estimates_test - real_trajectory)**2))
    
    plt.subplot(2, 2, i)
    plt.plot(t, real_trajectory, 'g--', label='Referencia', linewidth=2)
    plt.plot(t, measurements_noisy, 'b-', alpha=0.3, label='Ruidoso')
    plt.plot(t, estimates_test, 'r-', label='Filtrado', linewidth=2)
    plt.title(f'Alpha = {alpha} (RMSE: {error:.2f})')
    plt.xlabel('Tiempo (s)')
    plt.ylabel('Posición (m)')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8.5 Encontrar alpha óptimo

In [ ]:
# Encontrar alpha óptimo
optimal_alpha = ComplementaryFilter.find_optimal_alpha(
    measurements_noisy, measurements_integrated, real_trajectory
)

print(f"Alpha óptimo encontrado: {optimal_alpha:.2f}")

# Aplicar con alpha óptimo
cf_optimal = ComplementaryFilter(alpha=optimal_alpha)
estimates_optimal = cf_optimal.filter_series(measurements_noisy, measurements_integrated)

error_optimal = np.sqrt(np.mean((estimates_optimal - real_trajectory)**2))
print(f"Error RMS con alpha óptimo: {error_optimal:.2f}")

# RESUMEN FINAL

In [ ]:
print("\n" + "="*70)
print("RESUMEN DE RESULTADOS - PARTE C")
print("="*70)

print("\n✅ SECCIÓN 3: REGRESIÓN LINEAL")
print(f"   - Ecuación: {regression_model.get_equation()}")
print(f"   - R² Score: {metrics['r2']:.4f}")
print(f"   - RMSE: {metrics['rmse']:.4f}")
print(f"   - RSS: {rss:.4f}")

print("\n✅ SECCIÓN 4: DISTANCIA DEL COSENO")
print(f"   - Vectores analizados: 6")
print(f"   - Pares evaluados: 15")

print("\n✅ SECCIÓN 5: VISUALIZACIÓN 3D")
print(f"   - Puntos 3D cargados: {len(X_3d)}")
print(f"   - Centroide calculado: {centroide}")

print("\n✅ SECCIÓN 7: FILTRO DE KALMAN")
print(f"   - Mediciones filtradas: {len(estimates)}")
print(f"   - Error sin filtro (RMS): {np.sqrt(np.mean(error_measurements**2)):.2f}")
print(f"   - Error con filtro (RMS): {np.sqrt(np.mean(error_estimates[1:]**2)):.2f}")
print(f"   - Mejora: {mejora:.1f}%")

print("\n✅ SECCIÓN 8: FILTRO COMPLEMENTARIO")
print(f"   - Alpha óptimo: {optimal_alpha:.2f}")
print(f"   - Error RMS con alpha óptimo: {error_optimal:.2f}")

print("\n" + "="*70)